# SMA Report Analysis

Dieses Notebook erwartet, dass es direkt im heruntergeladenen SMA-Report-Ordner liegt.
Die CSV-Dateien werden aus `./data` geladen und mit kleinen Pandas-Helfern ausgewertet.

In [ ]:
from pathlib import Path
import math
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

REPORT_DIR = Path('.').resolve()
DATA_DIR = REPORT_DIR / 'data'
REPORT_DIR, DATA_DIR.exists()

In [ ]:
META_COLUMNS = {
    'timestamp',
    '__name__',
    'instance',
    'job',
    'layer',
    'unit',
    'treatment',
}


def list_csv_files(data_dir=DATA_DIR):
    return sorted(data_dir.glob('*.csv'))


def detect_value_columns(df):
    return [c for c in df.columns if c not in META_COLUMNS]


def load_metric(csv_file):
    df = pd.read_csv(csv_file)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    for col in detect_value_columns(df):
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_all_metrics(data_dir=DATA_DIR):
    return {csv_file.stem: load_metric(csv_file) for csv_file in list_csv_files(data_dir)}


def metric_info(frames):
    rows = []
    for name, df in frames.items():
        value_columns = detect_value_columns(df)
        rows.append({
            'metric': name,
            'rows': len(df),
            'value_columns': ', '.join(value_columns),
            'unit': df['unit'].dropna().iloc[0] if 'unit' in df.columns and not df['unit'].dropna().empty else None,
            'job': df['job'].dropna().iloc[0] if 'job' in df.columns and not df['job'].dropna().empty else None,
            'instance': df['instance'].dropna().iloc[0] if 'instance' in df.columns and not df['instance'].dropna().empty else None,
        })
    return pd.DataFrame(rows).sort_values('metric').reset_index(drop=True)


def metric_series(frames, metric_name):
    df = frames[metric_name].copy()
    value_columns = detect_value_columns(df)
    if not value_columns:
        raise ValueError(f'No value columns detected for {metric_name}')
    value_col = value_columns[-1]
    cols = [c for c in ['timestamp', 'treatment', value_col] if c in df.columns]
    return df[cols].rename(columns={value_col: 'value'})


def treatment_split(df):
    if 'treatment' not in df.columns:
        return {'all': df}
    return {str(name): group.copy() for name, group in df.groupby('treatment', dropna=False)}


def summarize_metric(frames, metric_name):
    series = metric_series(frames, metric_name)
    values = series['value'].dropna()
    if values.empty:
        return pd.Series({'metric': metric_name, 'count': 0})
    return pd.Series({
        'metric': metric_name,
        'count': int(values.count()),
        'min': float(values.min()),
        'max': float(values.max()),
        'mean': float(values.mean()),
        'median': float(values.median()),
        'last': float(values.iloc[-1]),
    })


def summarize_all(frames):
    rows = [summarize_metric(frames, metric_name) for metric_name in sorted(frames)]
    return pd.DataFrame(rows)


def plot_metric(frames, metric_name, ax=None, title=None):
    series = metric_series(frames, metric_name)
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    ax.plot(series['timestamp'], series['value'], marker='o', linewidth=1.5)
    ax.set_title(title or metric_name)
    ax.set_xlabel('timestamp')
    ax.set_ylabel('value')
    ax.grid(True, alpha=0.3)
    return ax


def plot_metrics_grid(frames, metric_names, cols=2, figsize=(14, 4)):
    rows = math.ceil(len(metric_names) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(figsize[0], figsize[1] * rows), squeeze=False)
    axes_flat = axes.flatten()
    for ax, metric_name in zip(axes_flat, metric_names):
        plot_metric(frames, metric_name, ax=ax)
    for ax in axes_flat[len(metric_names):]:
        ax.axis('off')
    fig.tight_layout()
    return fig, axes


def compare_treatment_metric(frames, metric_name):
    series = metric_series(frames, metric_name)
    if 'treatment' not in series.columns:
        return pd.DataFrame([{'treatment': 'all', 'mean': series['value'].mean(), 'max': series['value'].max(), 'last': series['value'].iloc[-1]}])
    rows = []
    for treatment, group in series.groupby('treatment', dropna=False):
        values = group['value'].dropna()
        rows.append({
            'treatment': treatment,
            'count': int(values.count()),
            'mean': float(values.mean()) if not values.empty else None,
            'max': float(values.max()) if not values.empty else None,
            'last': float(values.iloc[-1]) if not values.empty else None,
        })
    return pd.DataFrame(rows)


def metrics_by_prefix(frames, prefix):
    return sorted(name for name in frames if name.startswith(prefix))


def report_overview(frames):
    rows = []
    for metric_name, df in frames.items():
        timestamps = df['timestamp'].dropna() if 'timestamp' in df.columns else pd.Series(dtype='datetime64[ns, UTC]')
        values = metric_series(frames, metric_name)['value'].dropna()
        started_at = timestamps.min() if not timestamps.empty else pd.NaT
        finished_at = timestamps.max() if not timestamps.empty else pd.NaT
        duration_seconds = (finished_at - started_at).total_seconds() if pd.notna(started_at) and pd.notna(finished_at) else None
        rows.append({
            'metric': metric_name,
            'samples': int(values.count()),
            'started_at': started_at,
            'finished_at': finished_at,
            'duration_seconds': duration_seconds,
        })
    return pd.DataFrame(rows).sort_values(['started_at', 'metric']).reset_index(drop=True)


def resample_metric(frames, metric_name, rule='15s', agg='mean'):
    series = metric_series(frames, metric_name).dropna(subset=['timestamp']).copy()
    if series.empty:
        return series
    value = series.set_index('timestamp')['value'].resample(rule)
    if not hasattr(value, agg):
        raise ValueError(f'Unsupported aggregation: {agg}')
    resampled = getattr(value, agg)().dropna().reset_index()
    return resampled.rename(columns={'value': metric_name})


def merge_metrics(frames, metric_names, rule='15s', agg='mean'):
    merged = None
    for metric_name in metric_names:
        if metric_name not in frames:
            continue
        df = resample_metric(frames, metric_name, rule=rule, agg=agg)
        if df.empty:
            continue
        merged = df if merged is None else merged.merge(df, on='timestamp', how='outer')
    if merged is None:
        return pd.DataFrame(columns=['timestamp', *metric_names])
    return merged.sort_values('timestamp').reset_index(drop=True)


def correlate_metrics(frames, metric_names, rule='15s', agg='mean'):
    merged = merge_metrics(frames, metric_names, rule=rule, agg=agg)
    value_columns = [c for c in merged.columns if c != 'timestamp']
    if not value_columns:
        return pd.DataFrame()
    return merged[value_columns].corr(numeric_only=True)


def top_spikes(frames, metric_name, n=10):
    series = metric_series(frames, metric_name).copy()
    series = series.sort_values('value', ascending=False)
    return series.head(n).reset_index(drop=True)


def plot_treatments(frames, metric_name, ax=None):
    series = metric_series(frames, metric_name)
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 4))
    if 'treatment' not in series.columns:
        ax.plot(series['timestamp'], series['value'], marker='o', linewidth=1.5, label='all')
    else:
        for treatment, group in series.groupby('treatment', dropna=False):
            ax.plot(group['timestamp'], group['value'], marker='o', linewidth=1.5, label=str(treatment))
        ax.legend()
    ax.set_title(f'{metric_name} by treatment')
    ax.set_xlabel('timestamp')
    ax.set_ylabel('value')
    ax.grid(True, alpha=0.3)
    return ax


In [ ]:
frames = load_all_metrics()
metric_catalog = metric_info(frames)
metric_catalog

## Quick summaries

In [ ]:
summarize_all(frames).sort_values('metric').reset_index(drop=True)

## Report overview

In [ ]:
report_overview(frames)

## Beispiel: Elise GPU

In [ ]:
gpu_metrics = [
    name for name in [
        'elise_gpu_power_mw',
        'elise_gpu_temperature_celsius',
        'elise_gpu_utilization_percent',
    ] if name in frames
]
gpu_metrics

In [ ]:
if gpu_metrics:
    plot_metrics_grid(frames, gpu_metrics, cols=1, figsize=(12, 3.5))
    plt.show()
else:
    print('No Elise GPU metrics found in this report.')

## Beispiel: Agent, MCP client, MCP server

In [ ]:
interesting_metrics = [
    name for name in [
        'agent_cpu',
        'agent_memory',
        'mcp_client_cpu',
        'mcp_client_memory',
        'mcp_server_cpu',
        'mcp_server_memory',
        'mcp_client_avg_request_duration',
        'agent_avg_job_duration_ms',
    ] if name in frames
]
interesting_metrics

In [ ]:
if interesting_metrics:
    plot_metrics_grid(frames, interesting_metrics[:6], cols=2, figsize=(14, 3.5))
    plt.show()
else:
    print('No predefined agent/MCP metrics found in this report.')

## Nach Prefix filtern

In [ ]:
metrics_by_prefix(frames, 'agent_')

## Treatment comparison

In [ ]:
metric_name = 'agent_active_count'
if metric_name in frames:
    compare_treatment_metric(frames, metric_name)
else:
    print(f'{metric_name} not found in this report.')

## Korrelationen zwischen Metriken

In [ ]:
correlation_metrics = [
    name for name in [
        'agent_cpu',
        'agent_memory',
        'mcp_client_cpu',
        'mcp_server_cpu',
        'elise_gpu_utilization_percent',
    ] if name in frames
]
correlation_metrics

In [ ]:
if correlation_metrics:
    correlate_metrics(frames, correlation_metrics, rule='15s')
else:
    print('No predefined correlation metrics found in this report.')

## Freie Auswertung

In [ ]:
metric_name = 'mcp_server_tool_call_rate'
if metric_name in frames:
    display(metric_series(frames, metric_name).head())
    plot_metric(frames, metric_name)
    plt.show()
else:
    print(f'{metric_name} not found in this report.')

## Peaks einer Metrik

In [ ]:
metric_name = 'elise_gpu_power_mw'
if metric_name in frames:
    top_spikes(frames, metric_name, n=10)
else:
    print(f'{metric_name} not found in this report.')

## Plot nach Treatment

In [ ]:
metric_name = 'agent_active_count'
if metric_name in frames:
    plot_treatments(frames, metric_name)
    plt.show()
else:
    print(f'{metric_name} not found in this report.')